**Multi-Embedding Experiment Pipeline**

1. Generating text embeddings from different transformer models.
2. Combining embeddings with simple structured numeric features.
3. Training ML classifiers (Logistic Regression, Random Forest)
4. Comparing embedding models by re-running the pipeline.

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
# import required libraries
import re, numpy as np, pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sentence_transformers import SentenceTransformer

**Load a dataset - text and labels**

In [3]:
cats = ["sci.space", "rec.autos"]
ds = fetch_20newsgroups(subset="train", categories=cats,
                        remove=("headers","footers","quotes"))

X_text = pd.Series(ds.data)
y = pd.Series(ds.target)  # 0/1 labels

In [4]:
# view sample data point
X_text[0]

"Well thank you dennis for your as usual highly detailed and informative \nposting.   \n\nThe question i have about the proton, is  could it be  handled at\none of KSC's spare pads, without major  malfunction,  or could it be\nhandled at kourou  or Vandenberg?   \n\nNow if it uses storables,  then  how long would it take for the russians\nto equip something at cape york?\n\nIf  Proton were launched from a western site,  how would it compare to the\nT4/centaur?   As i see it, it should lift  very close to the T4."

**Create structured numeric features from raw text**

In [5]:
def text_struct_feats(s: str):
    """
    Convert a raw text string into simple numeric indicators.
    These represent *structured features* (non-textual).

    Examples:
    - Length, punctuation ratio, number of uppercase letters...
    - These help combine linguistic signals with embedding vectors.
    """
    s = s or ""

    n_chars = len(s)
    words = re.findall(r"\b\w+\b", s.lower())
    n_words = len(words)
    avg_word_len = (sum(len(w) for w in words) / (n_words or 1))

    punc = re.findall(r"[^\w\s]", s)
    punc_ratio = len(punc) / (n_chars or 1)

    digits = re.findall(r"\d", s)
    digit_ratio = len(digits) / (n_chars or 1)

    caps_ratio = sum(1 for c in s if c.isupper()) / (n_chars or 1)

    return pd.Series({
        "n_chars": n_chars,
        "n_words": n_words,
        "avg_word_len": avg_word_len,
        "punc_ratio": punc_ratio,
        "digit_ratio": digit_ratio,
        "caps_ratio": caps_ratio,
    })

In [6]:
# apply structured feature extraction
X_struct = X_text.apply(text_struct_feats)

NUMERIC_COLS = X_struct.columns.tolist()
CAT_COLS = []   # there are no categorical vars here, but pipeline supports them.

In [7]:
# Preprocess structured features (scale numeric)
preprocess = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS)
])

#### **Pipeline to use different embedding models and develop supervised models**

In [8]:
# embeddings models to test
embedding_models = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2",
    "intfloat/e5-small-v2",
    "BAAI/bge-small-en-v1.5"
]

# results store
results = []

In [9]:
'''Run experiment for each embedding model'''

for embed_model in embedding_models:
    print("\n" + "="*80)
    print(f"Computing embeddings using: {embed_model}")
    print("="*80)

    embedder = SentenceTransformer(embed_model)

    # E5 requires prefix for best results
    if "e5" in embed_model:
        processed_text = ["passage: " + t for t in X_text.fillna("").tolist()]
    else:
        processed_text = X_text.fillna("").tolist()

    # generate embeddings
    E = embedder.encode(processed_text, batch_size=64, convert_to_numpy=True)

    # preprocess structured features
    X_struct_proc = preprocess.fit_transform(X_struct)

    # horizontally concatenate numeric features and embedding vectors
    X_combined = np.hstack([X_struct_proc, E])

    # split data into train/test
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_combined, y.values, test_size=0.25,
        random_state=42, stratify=y
    )

    # supervised models
    models = {
        "LogisticRegression": LogisticRegression(max_iter=2000),
        "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42)
    }

    # train & evaluate
    for model_name, clf in models.items():
        clf.fit(X_tr, y_tr)
        pred = clf.predict(X_te)
        proba = clf.predict_proba(X_te)[:, 1]

        acc = accuracy_score(y_te, pred)
        f1 = f1_score(y_te, pred)
        auc = roc_auc_score(y_te, proba)

        results.append({
            "Embedding Model": embed_model,
            "Classifier": model_name,
            "Accuracy": acc,
            "F1 Score": f1,
            "ROC AUC": auc
        })

        print(f"\n--- Results for {embed_model} + {model_name} ---")
        print(f"Accuracy: {acc:.4f}")
        print(f"F1 Score: {f1:.4f}")
        print(f"ROC AUC: {auc:.4f}")



Computing embeddings using: sentence-transformers/all-MiniLM-L6-v2

--- Results for sentence-transformers/all-MiniLM-L6-v2 + LogisticRegression ---
Accuracy: 0.9394
F1 Score: 0.9388
ROC AUC: 0.9911

--- Results for sentence-transformers/all-MiniLM-L6-v2 + RandomForest ---
Accuracy: 0.9529
F1 Score: 0.9524
ROC AUC: 0.9736

Computing embeddings using: sentence-transformers/all-mpnet-base-v2

--- Results for sentence-transformers/all-mpnet-base-v2 + LogisticRegression ---
Accuracy: 0.9461
F1 Score: 0.9456
ROC AUC: 0.9939

--- Results for sentence-transformers/all-mpnet-base-v2 + RandomForest ---
Accuracy: 0.9596
F1 Score: 0.9592
ROC AUC: 0.9731

Computing embeddings using: intfloat/e5-small-v2

--- Results for intfloat/e5-small-v2 + LogisticRegression ---
Accuracy: 0.9495
F1 Score: 0.9495
ROC AUC: 0.9858

--- Results for intfloat/e5-small-v2 + RandomForest ---
Accuracy: 0.9663
F1 Score: 0.9664
ROC AUC: 0.9744

Computing embeddings using: BAAI/bge-small-en-v1.5

--- Results for BAAI/bge-s

In [13]:
# display final comparison table
print("\n\nFinal comparison table")
df_results = pd.DataFrame(results)
df_results.sort_values(by="ROC AUC", ascending=False)



Final comparison table


,Embedding Model,Classifier,Accuracy,F1 Score,ROC AUC
2,sentence-transformers/all-mpnet-base-v2,LogisticRegression,0.946128,0.945578,0.993878
0,sentence-transformers/all-MiniLM-L6-v2,LogisticRegression,0.939394,0.938776,0.991112
6,BAAI/bge-small-en-v1.5,LogisticRegression,0.952862,0.952381,0.990341
4,intfloat/e5-small-v2,LogisticRegression,0.949495,0.949495,0.985806
5,intfloat/e5-small-v2,RandomForest,0.966330,0.966443,0.974401
1,sentence-transformers/all-MiniLM-L6-v2,RandomForest,0.952862,0.952381,0.973585
3,sentence-transformers/all-mpnet-base-v2,RandomForest,0.959596,0.959184,0.973109
7,BAAI/bge-small-en-v1.5,RandomForest,0.939394,0.939597,0.970320


#### **Interpretations**

- The embeddings with simple structured features make “sci.space vs rec.autos” almost perfectly separable based on very high performance metrics.
- E5 embedding model with Random Forest gives the highest Accuracy/F1 (~0.97) which is best for strict classification performance.
- MPNet model with Logistic Regression yields the highest ROC AUC (~0.994), better for probability ranking and calibrated decisions.
- MPNet  model shows strongest discrimination power with the best ROC score.
- Logistic Regression consistently gives better ROC AUC, meaning cleaner linear separation and better-calibrated outputs.

